# Section 4: Evaluation and Interpretation
*Owner: Video lead*

**Needs from earlier sections:** `X_train`, `X_test`, `y_train`, `y_test`, `scaler`, `model`

The temporary cell below recreates the inputs and trains a model so you can start before the others are done. **Delete both temporary cells when merging.**

In [ ]:
# TEMPORARY STAND-IN: delete this whole cell after merging into the final notebook.
# It rebuilds what the Execution lead's section hands over, so you can work before it is ready.
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/Titanic-Dataset.csv")
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
df = pd.get_dummies(df, columns=["Embarked"], drop_first=True, dtype=int)

X = df.drop(columns="Survived")
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

In [ ]:
# TEMPORARY STAND-IN (part 2): delete after merging. In the final notebook, `model` comes from Section 3.
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, roc_auc_score)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

## 4.1 Evaluation metrics
Treat **survived (1)** as the "positive" class.

- **Accuracy:** fraction of all predictions that are correct.
- **Precision:** of the passengers predicted to survive, how many really survived.
- **Recall:** of the passengers who really survived, how many we found.
- **F1 score:** one number that balances precision and recall.

In [ ]:
metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1 score": f1_score(y_test, y_pred),
}
for name, value in metrics.items():
    print(f"{name:10s}: {value:.3f}")

## 4.2 Confusion matrix
Rows are the **actual** values and columns are the **predicted** values. The diagonal holds the correct predictions.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted: No", "Predicted: Yes"],
            yticklabels=["Actual: No", "Actual: Yes"])
plt.title("Confusion matrix")
plt.savefig("outputs/confusion_matrix.png", bbox_inches="tight")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Correctly predicted died: {tn}, correctly predicted survived: {tp}")
print(f"Wrongly predicted survived: {fp}, wrongly predicted died: {fn}")

## 4.3 Full report

In [ ]:
print(classification_report(y_test, y_pred, target_names=["Did not survive", "Survived"]))

## 4.4 ROC curve (optional)
The ROC curve shows how well the model separates the two classes at every possible threshold. The area under it (AUC) is 0.5 for random guessing and 1.0 for a perfect model.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.plot(fpr, tpr, label=f"Logistic regression (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], "--", label="Random guess")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve")
plt.legend()
plt.savefig("outputs/roc_curve.png", bbox_inches="tight")
plt.show()

## 4.5 Demo: predict for a new passenger
We enter the raw values for one passenger, in the **same column order** as the training data, apply the **same scaler**, then predict. Try changing the values (for example, a 30-year-old man in 3rd class).

In [ ]:
print("Column order:", X_train.columns.tolist())

# Order: Pclass, Sex (0=male, 1=female), Age, SibSp, Parch, Fare, Embarked_Q, Embarked_S
new_passenger = pd.DataFrame(
    [[1, 1, 28, 0, 0, 80.0, 0, 0]],      # TRY IT: change these values
    columns=X_train.columns
)

new_scaled = scaler.transform(new_passenger)
new_scaled = pd.DataFrame(new_scaled, columns=X_train.columns)

predicted_class = model.predict(new_scaled)[0]
probability = model.predict_proba(new_scaled)[0, 1]

print("Predicted class (0 = died, 1 = survived):", predicted_class)
print("Probability of survival:", round(probability, 2))

## 4.6 Interpretation of results
Write this using your own numbers (it is the most important part of the video):

1. **Overall performance:** The model reaches ___% accuracy on unseen passengers.
2. **Errors:** Look at the confusion matrix. Which mistake is more common, predicting survival for someone who died or the reverse? What do precision and recall say about it?
3. **Important features:** Using the coefficients from Section 3, which features matter most and in which direction? Does this match the EDA?
4. **Answer to the problem statement:** Can survival be predicted from these details? What are the limits of the model (few features, simple linear model, small dataset)?
5. **Possible improvements:** More features (for example, titles from names), other models, or tuning.